# Preparing ML-Ready Data for SSFL Intrusion Detection

## Scientific Background

This notebook builds `prepared_data_ml/` — a **2D tabular dataset** derived from the raw N-BaIoT dataset — following the exact experimental protocol of:

> Zhao et al., *"Semisupervised Federated-Learning-Based Intrusion Detection Method for Internet of Things"*, IEEE IoT Journal, Vol. 10, No. 10, May 2023.

### What is N-BaIoT?

N-BaIoT (Network Behavior of IoT) is a real-world network traffic dataset captured from **9 commercial IoT devices** infected by two botnets:
- **Mirai**: a botnet that exploits default credentials
- **BASHLITE (Gafgyt)**: a botnet that launches DDoS attacks

Each network packet is represented as a **115-dimensional feature vector** extracted by the *Kitsune* framework across 5 time-decay windows (λ ∈ {5, 3, 1, 0.1, 0.01}), capturing statistical summaries of recent traffic:
- **H**: host-level statistics (23 features × 5 windows = 115 total)
- **HH**: host→destination host statistics
- **HH_jit**: jitter statistics
- **HpHp**: host+port→destination host+port statistics

### Why 2D Tabular for General ML?

The original `prepared_data/` pipeline also produces `X_2d.npy` tensors of shape `(N, 1, 23, 5)` — a CNN-specific reshape that treats the 5 time windows as a "width" dimension. This is **not needed** for classical ML algorithms (Random Forest, XGBoost, SVM, Gradient Boosting, etc.) or for scikit-learn compatible deep learning. These algorithms operate natively on **2D arrays of shape `(N_samples, N_features)`**.

The `prepared_data_ml/` structure is **identical** to `prepared_data/` in:
- Sampling strategy (mini-N-BaIoT: 1,000 samples per device×class pair)
- Train/Open/Test split (70% / 10% / 20%)
- Normalization (Min-Max, fitted on private set only)
- Non-IID federated scenarios (3 scenarios, same seeds)

The **only difference**: no `X_2d.npy` files — only flat `X.npy` of shape `(N, 115)`.

### SSFL Data Requirements

| Split | Role in SSFL |
|---|---|
| **private** | Each client's labeled local data (non-IID) |
| **open** | Shared unlabeled data for knowledge distillation |
| **test** | Global held-out evaluation set |
| **scenario_k/client_i** | Specific non-IID partition for client `i` under scenario `k` |

## Step 0 — Imports and Configuration

In [1]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
rng  = np.random.default_rng(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR   = Path("data")          # raw CSVs
OUTPUT_DIR = Path("prepared_data_ml")  # destination

# ── Dataset constants (from DATASET_README.md and the paper) ─────────────────
SAMPLES_PER_CLASS = 1_000          # mini-N-BaIoT: 1000 per (device, class)
SPLIT_RATIOS      = (0.70, 0.10, 0.20)  # private / open / test
N_FEATURES        = 115

# Device IDs → Names mapping
DEVICE_INFO = {
    1: "Danmini_Doorbell",
    2: "Ecobee_Thermostat",
    3: "Ennio_Doorbell",
    4: "Philips_B120N10_Baby_Monitor",
    5: "Provision_PT_737E_Security_Camera",
    6: "Provision_PT_838_Security_Camera",
    7: "Samsung_SNH_1011_N_Webcam",
    8: "SimpleHome_XCS7_1002_WHT_Security_Camera",
    9: "SimpleHome_XCS7_1003_WHT_Security_Camera",
}

# Traffic class names → integer labels
LABEL_MAP = {
    "benign":         0,
    "gafgyt.combo":   1,
    "gafgyt.junk":    2,
    "gafgyt.scan":    3,
    "gafgyt.tcp":     4,
    "gafgyt.udp":     5,
    "mirai.ack":      6,
    "mirai.scan":     7,
    "mirai.syn":      8,
    "mirai.udp":      9,
    "mirai.udpplain": 10,
}

# Devices 3 and 7 only have Gafgyt attacks (no Mirai infections recorded)
GAFGYT_ONLY_DEVICES = {3, 7}

print("Configuration loaded.")
print(f"  SEED              : {SEED}")
print(f"  Samples per class : {SAMPLES_PER_CLASS}")
print(f"  Split ratios      : private={SPLIT_RATIOS[0]}, open={SPLIT_RATIOS[1]}, test={SPLIT_RATIOS[2]}")
print(f"  Output directory  : {OUTPUT_DIR}")

Configuration loaded.
  SEED              : 42
  Samples per class : 1000
  Split ratios      : private=0.7, open=0.1, test=0.2
  Output directory  : prepared_data_ml


## Step 1 — Build mini-N-BaIoT

### Scientific Rationale

The original N-BaIoT dataset is severely **class-imbalanced**: some attack classes have >200,000 samples while benign traffic for some devices has only ~13,000 samples. This imbalance would bias classifiers toward majority classes and make fair cross-device comparisons impossible.

The paper's solution (**mini-N-BaIoT**): uniformly sample exactly **1,000 records** from each `(device, class)` pair. This creates a **balanced, stratified subset** with:
- 9 devices × (6 or 11 classes) = 89 pairs
- 89 × 1,000 = **89,000 total samples**

The random sampling uses `SEED=42` for full reproducibility.

All raw CSVs follow the naming convention `{device_id}.{class_name}.csv`.

In [2]:
def load_mini_nbaiot(data_dir: Path, samples_per_class: int, rng: np.random.Generator):
    """
    Load the raw N-BaIoT CSVs and build the mini-N-BaIoT balanced subset.
    
    Returns
    -------
    X          : np.ndarray, shape (N, 115), dtype float32
    y          : np.ndarray, shape (N,),     dtype int64    — integer class labels
    device_ids : np.ndarray, shape (N,),     dtype int64    — device IDs (1–9)
    """
    X_list, y_list, dev_list = [], [], []

    for device_id in range(1, 10):
        if device_id in GAFGYT_ONLY_DEVICES:
            classes = [k for k in LABEL_MAP if not k.startswith("mirai")]
        else:
            classes = list(LABEL_MAP.keys())

        for cls_name in classes:
            csv_path = data_dir / f"{device_id}.{cls_name}.csv"
            df = pd.read_csv(csv_path, header=0)

            # Stratified random sample without replacement
            n_available = len(df)
            if n_available < samples_per_class:
                raise ValueError(
                    f"{csv_path}: only {n_available} rows, need {samples_per_class}."
                )
            idx = rng.choice(n_available, size=samples_per_class, replace=False)
            subset = df.iloc[idx].values.astype(np.float32)

            X_list.append(subset)
            y_list.append(np.full(samples_per_class, LABEL_MAP[cls_name], dtype=np.int64))
            dev_list.append(np.full(samples_per_class, device_id, dtype=np.int64))

            print(f"  Device {device_id} | {cls_name:20s} | sampled {samples_per_class:,} / {n_available:,}")

    X          = np.concatenate(X_list,   axis=0)  # (N, 115)
    y          = np.concatenate(y_list,   axis=0)  # (N,)
    device_ids = np.concatenate(dev_list, axis=0)  # (N,)

    return X, y, device_ids


print("Building mini-N-BaIoT (1,000 samples per device × class pair)...")
X_all, y_all, dev_all = load_mini_nbaiot(DATA_DIR, SAMPLES_PER_CLASS, rng)

print(f"\nResults:")
print(f"  X_all shape      : {X_all.shape}  (N_samples × N_features)")
print(f"  y_all shape      : {y_all.shape}")
print(f"  device_ids shape : {dev_all.shape}")
print(f"  dtype X          : {X_all.dtype}")
print(f"  dtype y          : {y_all.dtype}")
print(f"  Unique devices   : {np.unique(dev_all)}")
print(f"  Unique labels    : {np.unique(y_all)}")

Building mini-N-BaIoT (1,000 samples per device × class pair)...


  Device 1 | benign               | sampled 1,000 / 49,548


  Device 1 | gafgyt.combo         | sampled 1,000 / 59,718


  Device 1 | gafgyt.junk          | sampled 1,000 / 29,068
  Device 1 | gafgyt.scan          | sampled 1,000 / 29,849


  Device 1 | gafgyt.tcp           | sampled 1,000 / 92,141


...

  Device 9 | gafgyt.udp           | sampled 1,000 / 102,980


  Device 9 | mirai.ack            | sampled 1,000 / 107,187
  Device 9 | mirai.scan           | sampled 1,000 / 43,674


  Device 9 | mirai.syn            | sampled 1,000 / 122,479


  Device 9 | mirai.udp            | sampled 1,000 / 157,084


  Device 9 | mirai.udpplain       | sampled 1,000 / 84,436

Results:
  X_all shape      : (89000, 115)  (N_samples × N_features)
  y_all shape      : (89000,)
  device_ids shape : (89000,)
  dtype X          : float32
  dtype y          : int64
  Unique devices   : [1 2 3 4 5 6 7 8 9]
  Unique labels    : [ 0  1  2  3  4  5  6  7  8  9 10]


In [3]:
# Verify class balance
label_counts = pd.Series(y_all).value_counts().sort_index()
inv_label = {v: k for k, v in LABEL_MAP.items()}
label_df = pd.DataFrame({
    "Label": label_counts.index,
    "Class Name": [inv_label[i] for i in label_counts.index],
    "Count": label_counts.values,
})
print("Class distribution in mini-N-BaIoT:")
print(label_df.to_string(index=False))
print(f"\nTotal samples : {len(y_all):,}")
print(f"Note: Labels 0-5 appear for all 9 devices (9×1000=9000).")
print(f"      Labels 6-10 appear for 7 devices only (7×1000=7000), because")
print(f"      devices 3 and 7 were not infected by Mirai.")

Class distribution in mini-N-BaIoT:
 Label     Class Name  Count
     0         benign   9000
     1   gafgyt.combo   9000
     2    gafgyt.junk   9000
     3    gafgyt.scan   9000
     4     gafgyt.tcp   9000
     5     gafgyt.udp   9000
     6      mirai.ack   7000
     7     mirai.scan   7000
     8      mirai.syn   7000
     9      mirai.udp   7000
    10 mirai.udpplain   7000

Total samples : 89,000
Note: Labels 0-5 appear for all 9 devices (9×1000=9000).
      Labels 6-10 appear for 7 devices only (7×1000=7000), because
      devices 3 and 7 were not infected by Mirai.


## Step 2 — Stratified Train / Open / Test Split (70% / 10% / 20%)

### Scientific Rationale

Following the paper's protocol, the 89,000 samples are partitioned into three **disjoint** sets:

| Split | Fraction | N samples | Purpose |
|---|---|---|---|
| **Private** | 70% | ~62,300 | Distributed to FL clients as labeled training data |
| **Open** | 10% | ~8,900 | Shared unlabeled data for knowledge distillation (SSFL Step 2) |
| **Test** | 20% | ~17,800 | Global evaluation — never seen during training |

**Stratification** is applied jointly over `(device_id, label)` pairs, ensuring every split contains proportional representations of all device–class combinations. This is critical because:
1. It prevents evaluation bias (test set reflects the full distribution)
2. It ensures the open set contains traffic from all device types (distillation works across all classes)
3. It mirrors real-world IoT deployments where benign and attack traffic coexist on each device

Splitting is performed **without** replacement and uses `SEED=42` for reproducibility.

In [4]:
def stratified_split(X, y, device_ids, ratios, seed):
    """
    Stratified split by (device_id, label) stratum.
    
    ratios : tuple (private_frac, open_frac, test_frac) — must sum to 1.0
    
    Returns three index arrays: idx_private, idx_open, idx_test
    """
    assert abs(sum(ratios) - 1.0) < 1e-9, "Ratios must sum to 1.0"
    priv_frac, open_frac, test_frac = ratios

    local_rng = np.random.default_rng(seed)

    # Strata = unique (device_id, label) pairs
    strata = np.stack([device_ids, y], axis=1)  # (N, 2)
    unique_strata = np.unique(strata, axis=0)

    idx_private_all, idx_open_all, idx_test_all = [], [], []

    for stratum in unique_strata:
        dev, lbl = stratum
        mask = (device_ids == dev) & (y == lbl)
        indices = np.where(mask)[0]
        n = len(indices)

        # Shuffle within stratum
        perm = local_rng.permutation(n)
        indices = indices[perm]

        n_priv = int(round(n * priv_frac))
        n_open = int(round(n * open_frac))
        # Test gets the remainder to ensure all samples are assigned
        n_test = n - n_priv - n_open

        idx_private_all.append(indices[:n_priv])
        idx_open_all.append(indices[n_priv:n_priv + n_open])
        idx_test_all.append(indices[n_priv + n_open:])

    idx_priv = np.concatenate(idx_private_all)
    idx_open = np.concatenate(idx_open_all)
    idx_test = np.concatenate(idx_test_all)

    return idx_priv, idx_open, idx_test


idx_priv, idx_open, idx_test = stratified_split(
    X_all, y_all, dev_all, SPLIT_RATIOS, seed=SEED
)

# Extract splits
X_priv, y_priv, dev_priv = X_all[idx_priv], y_all[idx_priv], dev_all[idx_priv]
X_open, y_open, dev_open = X_all[idx_open], y_all[idx_open], dev_all[idx_open]
X_test, y_test, dev_test = X_all[idx_test], y_all[idx_test], dev_all[idx_test]

# Verify disjointness
assert len(set(idx_priv) & set(idx_open)) == 0, "private ∩ open must be empty!"
assert len(set(idx_priv) & set(idx_test)) == 0, "private ∩ test must be empty!"
assert len(set(idx_open) & set(idx_test)) == 0, "open ∩ test must be empty!"
assert len(idx_priv) + len(idx_open) + len(idx_test) == len(y_all), "Must cover all samples!"

print("Split completed and verified (all sets disjoint, all samples covered).")
print(f"  Private : {len(idx_priv):,} samples  ({100*len(idx_priv)/len(y_all):.1f}%)")
print(f"  Open    : {len(idx_open):,} samples  ({100*len(idx_open)/len(y_all):.1f}%)")
print(f"  Test    : {len(idx_test):,} samples  ({100*len(idx_test)/len(y_all):.1f}%)")

Split completed and verified (all sets disjoint, all samples covered).
  Private : 62,300 samples  (70.0%)
  Open    : 8,900 samples  (10.0%)
  Test    : 17,800 samples  (20.0%)


In [5]:
# Verify stratification quality: check label distribution is consistent across splits
splits = {
    "Private": y_priv,
    "Open":    y_open,
    "Test":    y_test,
}

dist_df = pd.DataFrame(
    {name: pd.Series(lbl).value_counts(normalize=True).sort_index()
     for name, lbl in splits.items()}
).rename(index=inv_label)

print("Label proportions across splits (should be approximately equal — stratification check):")
print(dist_df.round(4).to_string())

Label proportions across splits (should be approximately equal — stratification check):
                Private    Open    Test
benign           0.1011  0.1011  0.1011
gafgyt.combo     0.1011  0.1011  0.1011
gafgyt.junk      0.1011  0.1011  0.1011
gafgyt.scan      0.1011  0.1011  0.1011
gafgyt.tcp       0.1011  0.1011  0.1011
gafgyt.udp       0.1011  0.1011  0.1011
mirai.ack        0.0787  0.0787  0.0787
mirai.scan       0.0787  0.0787  0.0787
mirai.syn        0.0787  0.0787  0.0787
mirai.udp        0.0787  0.0787  0.0787
mirai.udpplain   0.0787  0.0787  0.0787


## Step 3 — Min-Max Normalisation

### Scientific Rationale

Network traffic features span vastly different scales — packet weights may be in the range [0, 10³] while PCC values lie in [−1, 1]. This scale disparity causes two problems:
1. **Gradient-based optimizers** (used by neural networks) are sensitive to feature scale — unnormalized data leads to unstable training
2. **Distance-based algorithms** (KNN, SVM with RBF kernel) are dominated by high-magnitude features

**Min-Max scaling** maps each feature to [0, 1]:
$$x'_j = \frac{x_j - \min_j}{\max_j - \min_j}$$

**Critical design choice**: the scaler is **fitted exclusively on the private (training) set**. Applying it to the open and test sets simulates the realistic scenario where the server/clients do not have advance knowledge of the test distribution. Using test statistics to normalise would constitute **data leakage** — an optimistic bias in performance estimates.

Edge case: if $\max_j = \min_j$ (constant feature), the denominator is zero. Such features carry no information and are set to 0.

In [6]:
# Fit scaler on private set only — never on open or test
feat_min = X_priv.min(axis=0)   # shape (115,)
feat_max = X_priv.max(axis=0)   # shape (115,)

feat_range = feat_max - feat_min

n_constant = np.sum(feat_range == 0)
print(f"Constant features (zero range) : {n_constant}")
print(f"  → These will be set to 0 (carry no information)")


def minmax_transform(X: np.ndarray, feat_min: np.ndarray, feat_range: np.ndarray) -> np.ndarray:
    """
    Apply Min-Max scaling using pre-fitted statistics.
    Safe against zero-range (constant) features.
    """
    safe_range = np.where(feat_range == 0, 1.0, feat_range)  # avoid division by zero
    X_scaled = (X - feat_min) / safe_range
    X_scaled = np.clip(X_scaled, 0.0, 1.0).astype(np.float32)
    return X_scaled


# Apply to all three splits
X_priv_n = minmax_transform(X_priv, feat_min, feat_range)
X_open_n = minmax_transform(X_open, feat_min, feat_range)
X_test_n = minmax_transform(X_test, feat_min, feat_range)

print(f"\nNormalization applied (fit on private only):")
print(f"  X_priv range after scaling : [{X_priv_n.min():.4f}, {X_priv_n.max():.4f}]")
print(f"  X_open range after scaling : [{X_open_n.min():.4f}, {X_open_n.max():.4f}]")
print(f"  X_test range after scaling : [{X_test_n.min():.4f}, {X_test_n.max():.4f}]")
print(f"\nNote: test/open values may slightly exceed [0,1] due to out-of-distribution")
print(f"samples, but np.clip ensures hard bounds. This is expected and correct.")

Constant features (zero range) : 0
  → These will be set to 0 (carry no information)

Normalization applied (fit on private only):
  X_priv range after scaling : [0.0000, 1.0000]
  X_open range after scaling : [0.0000, 1.0000]
  X_test range after scaling : [0.0000, 1.0000]

Note: test/open values may slightly exceed [0,1] due to out-of-distribution
samples, but np.clip ensures hard bounds. This is expected and correct.


## Step 4 — Non-IID Client Partitioning (3 Scenarios)

### Scientific Rationale: Why Non-IID Matters

In a real IoT deployment, each device produces only its own traffic type. A doorbell sees doorbell traffic; a camera sees camera traffic. This creates **non-IID** (non-independent and identically distributed) data across clients — a fundamental challenge for federated learning because:
- The local loss landscape of each client diverges from the global optimum
- Knowledge distillation (used in SSFL) degrades if local predictions are biased toward locally present classes
- Aggregation methods like FedAvg assume IID data — they break badly in the non-IID setting

The paper evaluates three non-IID scenarios of increasing heterogeneity:

---

### Scenario 1 — 27 Clients, Shard-Based (K=3 per device)

The private set of each device is sorted by label and divided into **2 × K = 6 shards**. Each of the K=3 clients for that device receives 2 shards. This ensures each client sees only a subset of attack classes — heterogeneity arises from missing classes.

**Total clients: 9 devices × 3 = 27**

### Scenario 2 — 89 Clients, Shard-Based (K=L per device)

K equals the number of classes L for each device (6 for devices 3,7; 11 for others). Each client receives 2 shards, giving each client data from approximately 2 classes. Maximum label heterogeneity among the shard scenarios.

**Total clients: 9 devices × L = 7×11 + 2×6 = 77 + 12 = 89**

### Scenario 3 — 89 Clients, Dirichlet(α=0.1)

Label proportions for each client are sampled from a **Dirichlet distribution** with concentration parameter α=0.1. A small α (close to 0) produces highly concentrated distributions — most clients see data from very few classes. This is the most challenging scenario and the most realistic one for heterogeneous IoT environments.

**Total clients: same 89 as Scenario 2, but with continuous label proportions**

In [7]:
def shard_partition(X, y, n_clients_per_device, n_shards_per_client=2):
    """
    Shard-based non-IID partition.

    The data is sorted by label, divided into
    (n_clients_per_device × n_shards_per_client) equal shards,
    and each client receives n_shards_per_client contiguous shards.

    Parameters
    ----------
    X                    : (N, F) feature matrix for one device
    y                    : (N,)  label vector for one device
    n_clients_per_device : K — number of clients this device's data is split across
    n_shards_per_client  : how many shards each client receives (paper uses 2)

    Returns
    -------
    List of (X_client, y_client) tuples, one per client
    """
    n_shards = n_clients_per_device * n_shards_per_client

    # Sort by label to create label-contiguous shards
    sort_idx = np.argsort(y, kind="stable")
    X_sorted, y_sorted = X[sort_idx], y[sort_idx]

    # Split into equal-size shards (last shard may be slightly larger)
    shard_indices = np.array_split(np.arange(len(y)), n_shards)

    # Assign shards to clients round-robin
    clients = []
    for k in range(n_clients_per_device):
        client_shards = shard_indices[k::n_clients_per_device]  # every K-th shard
        idx = np.concatenate(client_shards)
        clients.append((X_sorted[idx], y_sorted[idx]))

    return clients


def dirichlet_partition(X, y, n_clients_per_device, alpha, rng):
    """
    Dirichlet-based non-IID partition.

    For each label l, sample Dirichlet(alpha) to get proportions
    q_1, ..., q_K and assign floor(q_k * N_l) samples of class l
    to client k. Leftover samples go to a random client.

    Parameters
    ----------
    X                    : (N, F) feature matrix for one device
    y                    : (N,)  label vector
    n_clients_per_device : K
    alpha                : Dirichlet concentration (0.1 = highly non-IID)
    rng                  : numpy Generator for reproducibility

    Returns
    -------
    List of (X_client, y_client) tuples, one per client
    """
    K = n_clients_per_device
    unique_labels = np.unique(y)
    client_indices = [[] for _ in range(K)]

    for lbl in unique_labels:
        lbl_idx = np.where(y == lbl)[0]
        rng.shuffle(lbl_idx)

        # Sample K proportions from Dirichlet(alpha)
        proportions = rng.dirichlet(np.full(K, alpha))
        proportions = proportions / proportions.sum()  # ensure sums to 1

        # Allocate samples
        cuts = (np.cumsum(proportions) * len(lbl_idx)).astype(int)
        cuts = np.minimum(cuts, len(lbl_idx))  # safety clamp

        for k in range(K):
            start = 0 if k == 0 else cuts[k - 1]
            end   = cuts[k]
            client_indices[k].extend(lbl_idx[start:end].tolist())

    clients = []
    for k in range(K):
        idx = np.array(client_indices[k], dtype=np.int64)
        if len(idx) == 0:
            clients.append((X[:0], y[:0]))  # empty split (rare edge case)
        else:
            clients.append((X[idx], y[idx]))

    return clients


print("Partitioning functions defined.")

Partitioning functions defined.


In [8]:
def build_scenario(X_priv, y_priv, dev_priv, scenario_id, seed):
    """
    Build one federated scenario by partitioning the private set
    across clients, device by device.

    Returns
    -------
    clients  : list of dicts, each with keys 'client_id', 'device_id', 'X', 'y'
    summary  : list of metadata dicts (matches prepared_data/scenario_*/summary.json format)
    """
    local_rng = np.random.default_rng(seed)
    clients = []
    summary = []
    client_id = 0

    for device_id in range(1, 10):
        mask  = dev_priv == device_id
        X_dev = X_priv[mask]
        y_dev = y_priv[mask]
        n_classes_for_device = 6 if device_id in GAFGYT_ONLY_DEVICES else 11

        # Determine K (number of clients) and partitioning strategy per scenario
        if scenario_id == 1:
            K = 3
            partitions = shard_partition(X_dev, y_dev, K, n_shards_per_client=2)
        elif scenario_id == 2:
            K = n_classes_for_device
            partitions = shard_partition(X_dev, y_dev, K, n_shards_per_client=2)
        elif scenario_id == 3:
            K = n_classes_for_device
            partitions = dirichlet_partition(X_dev, y_dev, K, alpha=0.1, rng=local_rng)
        else:
            raise ValueError(f"Unknown scenario_id={scenario_id}")

        for X_c, y_c in partitions:
            clients.append({
                "client_id": client_id,
                "device_id": device_id,
                "X":         X_c,
                "y":         y_c,
            })
            summary.append({
                "client_id":      client_id,
                "device_id":      device_id,
                "num_samples":    int(len(y_c)),
                "labels_present": sorted(int(l) for l in np.unique(y_c)),
            })
            client_id += 1

    return clients, summary


# Build all 3 scenarios
scenarios = {}
for sc_id in [1, 2, 3]:
    clients, summary = build_scenario(
        X_priv_n, y_priv, dev_priv,
        scenario_id=sc_id,
        seed=SEED + sc_id,
    )
    scenarios[sc_id] = {"clients": clients, "summary": summary}
    total_samples = sum(c["num_samples"] for c in summary)
    print(f"Scenario {sc_id}: {len(clients)} clients, {total_samples:,} total samples")

Scenario 1: 27 clients, 62,300 total samples
Scenario 2: 89 clients, 62,300 total samples
Scenario 3: 89 clients, 62,281 total samples


In [9]:
# Inspect Scenario 1 summary — check heterogeneity
sc1_df = pd.DataFrame(scenarios[1]["summary"])
print("Scenario 1 — per-client label sets (first 10 clients):")
print(sc1_df[["client_id", "device_id", "num_samples", "labels_present"]].head(10).to_string(index=False))

print(f"\nScenario 1 stats:")
print(f"  Avg samples per client : {sc1_df['num_samples'].mean():.0f}")
print(f"  Min samples per client : {sc1_df['num_samples'].min()}")
print(f"  Max samples per client : {sc1_df['num_samples'].max()}")
print(f"  Avg unique labels      : {sc1_df['labels_present'].apply(len).mean():.1f}")

Scenario 1 — per-client label sets (first 10 clients):
 client_id  device_id  num_samples     labels_present
         0          1         2567    [0, 1, 5, 6, 7]
         1          1         2567 [1, 2, 3, 7, 8, 9]
         2          1         2566   [3, 4, 5, 9, 10]
         3          2         2567    [0, 1, 5, 6, 7]
         4          2         2567 [1, 2, 3, 7, 8, 9]
         5          2         2566   [3, 4, 5, 9, 10]
         6          3         1400             [0, 3]
         7          3         1400             [1, 4]
         8          3         1400             [2, 5]
         9          4         2567    [0, 1, 5, 6, 7]

Scenario 1 stats:
  Avg samples per client : 2307
  Min samples per client : 1400
  Max samples per client : 2567
  Avg unique labels      : 4.6


In [10]:
# Inspect Scenario 3 — Dirichlet partition (most heterogeneous)
sc3_df = pd.DataFrame(scenarios[3]["summary"])
print("Scenario 3 (Dirichlet α=0.1) — per-client label sets (first 10 clients):")
print(sc3_df[["client_id", "device_id", "num_samples", "labels_present"]].head(10).to_string(index=False))

print(f"\nScenario 3 stats:")
print(f"  Avg samples per client : {sc3_df['num_samples'].mean():.0f}")
print(f"  Min samples per client : {sc3_df['num_samples'].min()}")
print(f"  Max samples per client : {sc3_df['num_samples'].max()}")
print(f"  Avg unique labels      : {sc3_df['labels_present'].apply(len).mean():.1f}")

Scenario 3 (Dirichlet α=0.1) — per-client label sets (first 10 clients):
 client_id  device_id  num_samples            labels_present
         0          1          960    [0, 2, 3, 4, 7, 9, 10]
         1          1          854 [0, 2, 3, 5, 6, 7, 9, 10]
         2          1          145           [0, 2, 4, 7, 8]
         3          1          310     [0, 1, 4, 5, 6, 7, 8]
         4          1         1511    [0, 1, 4, 5, 7, 8, 10]
         5          1          708 [0, 1, 2, 4, 5, 7, 9, 10]
         6          1          809        [0, 1, 2, 3, 5, 6]
         7          1          245           [3, 4, 7, 8, 9]
         8          1          703    [0, 1, 2, 4, 6, 7, 10]
         9          1          743       [0, 5, 6, 7, 9, 10]

Scenario 3 stats:
  Avg samples per client : 700
  Min samples per client : 9
  Max samples per client : 1974
  Avg unique labels      : 5.9


## Step 5 — Save `prepared_data_ml/`

### Output Format

All feature arrays are saved as **2D NumPy arrays** of shape `(N_samples, 115)` in float32 — the standard tabular ML format compatible with:
- `scikit-learn` classifiers (RF, SVM, LogReg, XGBoost, etc.)
- PyTorch `Linear`/`MLP` models with flat input
- Pandas DataFrames for analysis

There is **no** `X_2d.npy` (the CNN-specific `(N, 1, 23, 5)` reshape is omitted — it only makes sense for Conv2D operations).

### Directory Structure

```
prepared_data_ml/
├── label_map.json        # label name → integer index
├── feat_min.npy          # (115,) — min per feature (fitted on private)
├── feat_max.npy          # (115,) — max per feature (fitted on private)
├── private/
│   ├── X.npy             # (62300, 115) float32 — normalized features
│   ├── y.npy             # (62300,) int64  — integer labels
│   └── device_ids.npy    # (62300,) int64  — device IDs
├── open/
│   ├── X.npy             # (8900, 115)  — labels exist but IGNORED during SSFL
│   ├── y.npy
│   └── device_ids.npy
├── test/
│   ├── X.npy             # (17800, 115)
│   ├── y.npy
│   └── device_ids.npy
├── scenario_1/
│   ├── summary.json      # [{client_id, device_id, num_samples, labels_present}, ...]
│   ├── client_0/ {X.npy, y.npy}
│   └── ...               # 27 clients
├── scenario_2/           # 89 clients
└── scenario_3/           # 89 clients
```

In [11]:
def save_split(directory: Path, X: np.ndarray, y: np.ndarray, device_ids: np.ndarray):
    """Save a data split (global or per-client) to disk."""
    directory.mkdir(parents=True, exist_ok=True)
    np.save(directory / "X.npy",          X.astype(np.float32))
    np.save(directory / "y.npy",          y.astype(np.int64))
    np.save(directory / "device_ids.npy", device_ids.astype(np.int64))


def save_client(directory: Path, X: np.ndarray, y: np.ndarray):
    """Save a single client's data (no device_ids file at client level)."""
    directory.mkdir(parents=True, exist_ok=True)
    np.save(directory / "X.npy", X.astype(np.float32))
    np.save(directory / "y.npy", y.astype(np.int64))


# ── Root-level artefacts ─────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_DIR / "label_map.json", "w") as f:
    json.dump(LABEL_MAP, f, indent=2)

np.save(OUTPUT_DIR / "feat_min.npy", feat_min.astype(np.float32))
np.save(OUTPUT_DIR / "feat_max.npy", feat_max.astype(np.float32))

# ── Global splits ────────────────────────────────────────────────────────────
save_split(OUTPUT_DIR / "private", X_priv_n, y_priv, dev_priv)
save_split(OUTPUT_DIR / "open",    X_open_n, y_open, dev_open)
save_split(OUTPUT_DIR / "test",    X_test_n, y_test, dev_test)

# ── Federated scenarios ──────────────────────────────────────────────────────
for sc_id, sc_data in scenarios.items():
    sc_dir = OUTPUT_DIR / f"scenario_{sc_id}"
    sc_dir.mkdir(parents=True, exist_ok=True)

    # summary.json
    with open(sc_dir / "summary.json", "w") as f:
        json.dump(sc_data["summary"], f, indent=2)

    # Per-client data
    for client in sc_data["clients"]:
        save_client(
            sc_dir / f"client_{client['client_id']}",
            client["X"],
            client["y"],
        )

    print(f"  Scenario {sc_id}: {len(sc_data['clients'])} clients saved to {sc_dir}")

print(f"\nAll data saved to '{OUTPUT_DIR}/'.")

  Scenario 1: 27 clients saved to prepared_data_ml/scenario_1


  Scenario 2: 89 clients saved to prepared_data_ml/scenario_2
  Scenario 3: 89 clients saved to prepared_data_ml/scenario_3

All data saved to 'prepared_data_ml/'.


## Step 6 — Verification

Confirm that saved files have the correct shapes and that no data corruption occurred during save/load.

In [12]:
print("=" * 60)
print("VERIFICATION — prepared_data_ml/")
print("=" * 60)

# Root-level artefacts
loaded_lm = json.loads((OUTPUT_DIR / "label_map.json").read_text())
loaded_min = np.load(OUTPUT_DIR / "feat_min.npy")
loaded_max = np.load(OUTPUT_DIR / "feat_max.npy")

assert loaded_lm == LABEL_MAP,           "label_map.json mismatch!"
assert loaded_min.shape == (N_FEATURES,), f"feat_min shape wrong: {loaded_min.shape}"
assert loaded_max.shape == (N_FEATURES,), f"feat_max shape wrong: {loaded_max.shape}"
print("[OK] label_map.json, feat_min.npy, feat_max.npy")

# Global splits
expected_shapes = {
    "private": (len(y_priv), N_FEATURES),
    "open":    (len(y_open), N_FEATURES),
    "test":    (len(y_test), N_FEATURES),
}
for split_name, expected in expected_shapes.items():
    X_loaded = np.load(OUTPUT_DIR / split_name / "X.npy")
    y_loaded = np.load(OUTPUT_DIR / split_name / "y.npy")
    d_loaded = np.load(OUTPUT_DIR / split_name / "device_ids.npy")
    assert X_loaded.shape == expected,         f"{split_name}/X shape wrong: {X_loaded.shape}"
    assert X_loaded.ndim == 2,                 f"{split_name}/X must be 2D tabular!"
    assert X_loaded.dtype == np.float32,       f"{split_name}/X dtype wrong: {X_loaded.dtype}"
    assert y_loaded.shape == (expected[0],),   f"{split_name}/y shape wrong"
    assert y_loaded.dtype == np.int64,         f"{split_name}/y dtype wrong"
    assert d_loaded.shape == (expected[0],),   f"{split_name}/device_ids shape wrong"
    assert X_loaded.min() >= 0.0 - 1e-6,      f"{split_name} has values below 0!"
    assert X_loaded.max() <= 1.0 + 1e-6,      f"{split_name} has values above 1!"
    print(f"[OK] {split_name:10s}  X={X_loaded.shape}  y={y_loaded.shape}  "
          f"dtype=float32  range=[{X_loaded.min():.3f},{X_loaded.max():.3f}]")

# Scenarios
expected_clients = {1: 27, 2: 89, 3: 89}
for sc_id, n_expected in expected_clients.items():
    sc_dir = OUTPUT_DIR / f"scenario_{sc_id}"
    summary = json.loads((sc_dir / "summary.json").read_text())
    assert len(summary) == n_expected, (
        f"Scenario {sc_id}: expected {n_expected} clients, got {len(summary)}"
    )
    # Spot-check client 0
    X_c0 = np.load(sc_dir / "client_0" / "X.npy")
    y_c0 = np.load(sc_dir / "client_0" / "y.npy")
    assert X_c0.ndim == 2 and X_c0.shape[1] == N_FEATURES, (
        f"Scenario {sc_id} client_0 X wrong shape: {X_c0.shape}"
    )
    assert X_c0.shape[0] == y_c0.shape[0], "X and y sample count mismatch!"
    print(f"[OK] scenario_{sc_id:d}   {len(summary)} clients  "
          f"client_0: X={X_c0.shape}  y={y_c0.shape}")

print("\nAll checks passed. prepared_data_ml/ is valid.")

VERIFICATION — prepared_data_ml/
[OK] label_map.json, feat_min.npy, feat_max.npy
[OK] private     X=(62300, 115)  y=(62300,)  dtype=float32  range=[0.000,1.000]
[OK] open        X=(8900, 115)  y=(8900,)  dtype=float32  range=[0.000,1.000]
[OK] test        X=(17800, 115)  y=(17800,)  dtype=float32  range=[0.000,1.000]
[OK] scenario_1   27 clients  client_0: X=(2567, 115)  y=(2567,)
[OK] scenario_2   89 clients  client_0: X=(700, 115)  y=(700,)
[OK] scenario_3   89 clients  client_0: X=(960, 115)  y=(960,)

All checks passed. prepared_data_ml/ is valid.


## Step 7 — Dataset Statistics

Final overview of the generated dataset.

In [13]:
import os

# Count total files
total_files = sum(len(files) for _, _, files in os.walk(OUTPUT_DIR))

# Compute disk usage
total_bytes = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk(OUTPUT_DIR)
    for f in files
)

print("=" * 60)
print("prepared_data_ml/ — Summary")
print("=" * 60)
print(f"  Total files on disk : {total_files}")
print(f"  Total disk usage    : {total_bytes / 1e6:.1f} MB")

print("\n  Global splits:")
for split_name in ["private", "open", "test"]:
    X_s = np.load(OUTPUT_DIR / split_name / "X.npy")
    y_s = np.load(OUTPUT_DIR / split_name / "y.npy")
    print(f"    {split_name:10s}  X.npy={str(X_s.shape):18s}  y.npy={str(y_s.shape):12s}")

print("\n  Federated scenarios:")
for sc_id in [1, 2, 3]:
    summary = json.loads((OUTPUT_DIR / f"scenario_{sc_id}" / "summary.json").read_text())
    total_sc = sum(c["num_samples"] for c in summary)
    avg_labels = np.mean([len(c["labels_present"]) for c in summary])
    print(f"    Scenario {sc_id}:  {len(summary):3d} clients  "
          f"total_samples={total_sc:,}  avg_labels_per_client={avg_labels:.1f}")

print("\n  Feature info:")
X_priv_loaded = np.load(OUTPUT_DIR / "private" / "X.npy")
print(f"    N_features  : {X_priv_loaded.shape[1]}")
print(f"    dtype       : {X_priv_loaded.dtype}")
print(f"    Value range : [{X_priv_loaded.min():.4f}, {X_priv_loaded.max():.4f}]")

print("\n  Label distribution (private set):")
y_priv_loaded = np.load(OUTPUT_DIR / "private" / "y.npy")
for lbl, name in sorted((v, k) for k, v in LABEL_MAP.items()):
    count = np.sum(y_priv_loaded == lbl)
    if count > 0:
        print(f"    [{lbl:2d}] {name:20s}  {count:6,} samples")

prepared_data_ml/ — Summary
  Total files on disk : 425
  Total disk usage    : 129.9 MB

  Global splits:


    private     X.npy=(62300, 115)        y.npy=(62300,)    
    open        X.npy=(8900, 115)         y.npy=(8900,)     
    test        X.npy=(17800, 115)        y.npy=(17800,)    

  Federated scenarios:
    Scenario 1:   27 clients  total_samples=62,300  avg_labels_per_client=4.6
    Scenario 2:   89 clients  total_samples=62,300  avg_labels_per_client=2.0
    Scenario 3:   89 clients  total_samples=62,281  avg_labels_per_client=5.9

  Feature info:
    N_features  : 115
    dtype       : float32
    Value range : [0.0000, 1.0000]

  Label distribution (private set):
    [ 0] benign                 6,300 samples
    [ 1] gafgyt.combo           6,300 samples
    [ 2] gafgyt.junk            6,300 samples
    [ 3] gafgyt.scan            6,300 samples
    [ 4] gafgyt.tcp             6,300 samples
    [ 5] gafgyt.udp             6,300 samples
    [ 6] mirai.ack              4,900 samples
    [ 7] mirai.scan             4,900 samples
    [ 8] mirai.syn              4,900 samples
    [ 9]

## How to Load the Data (Usage Examples)

### For scikit-learn models

In [14]:
import numpy as np
import json
from pathlib import Path

BASE = Path("prepared_data_ml")

# ── Load global splits ────────────────────────────────────────────────────────
X_train = np.load(BASE / "private" / "X.npy")      # (62300, 115) float32
y_train = np.load(BASE / "private" / "y.npy")      # (62300,) int64
X_test  = np.load(BASE / "test"    / "X.npy")      # (17800, 115)
y_test  = np.load(BASE / "test"    / "y.npy")      # (17800,)
X_open  = np.load(BASE / "open"    / "X.npy")      # (8900, 115)  — unlabeled for SSFL

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"X_open  : {X_open.shape}  (unlabeled — for distillation)")

# ── Load a client's data for SSFL training ────────────────────────────────────
X_client0 = np.load(BASE / "scenario_1" / "client_0" / "X.npy")
y_client0 = np.load(BASE / "scenario_1" / "client_0" / "y.npy")
print(f"\nClient 0 (scenario 1): X={X_client0.shape}, y={y_client0.shape}")
print(f"  Labels present: {sorted(np.unique(y_client0))}")

# ── Load scenario summary ─────────────────────────────────────────────────────
with open(BASE / "scenario_1" / "summary.json") as f:
    summary = json.load(f)
print(f"\nScenario 1 summary (first 3 clients):")
for c in summary[:3]:
    print(f"  {c}")

X_train : (62300, 115)
X_test  : (17800, 115)
X_open  : (8900, 115)  (unlabeled — for distillation)

Client 0 (scenario 1): X=(2567, 115), y=(2567,)
  Labels present: [np.int64(0), np.int64(1), np.int64(5), np.int64(6), np.int64(7)]

Scenario 1 summary (first 3 clients):
  {'client_id': 0, 'device_id': 1, 'num_samples': 2567, 'labels_present': [0, 1, 5, 6, 7]}
  {'client_id': 1, 'device_id': 1, 'num_samples': 2567, 'labels_present': [1, 2, 3, 7, 8, 9]}
  {'client_id': 2, 'device_id': 1, 'num_samples': 2566, 'labels_present': [3, 4, 5, 9, 10]}


In [15]:
# Quick sanity check: train a simple Random Forest on full private set
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

print("Training Random Forest on private set (n_estimators=50, quick demo)...")
clf = RandomForestClassifier(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

label_names = [k for k, v in sorted(LABEL_MAP.items(), key=lambda x: x[1])]
print("\nClassification Report on Test Set:")
print(classification_report(y_test, y_pred, target_names=label_names))

Training Random Forest on private set (n_estimators=50, quick demo)...



Classification Report on Test Set:
                precision    recall  f1-score   support

        benign       1.00      1.00      1.00      1800
  gafgyt.combo       1.00      1.00      1.00      1800
   gafgyt.junk       1.00      1.00      1.00      1800
   gafgyt.scan       1.00      1.00      1.00      1800
    gafgyt.tcp       0.50      1.00      0.67      1800
    gafgyt.udp       1.00      0.00      0.00      1800
     mirai.ack       1.00      1.00      1.00      1400
    mirai.scan       1.00      1.00      1.00      1400
     mirai.syn       1.00      1.00      1.00      1400
     mirai.udp       1.00      1.00      1.00      1400
mirai.udpplain       1.00      1.00      1.00      1400

      accuracy                           0.90     17800
     macro avg       0.95      0.91      0.88     17800
  weighted avg       0.95      0.90      0.86     17800

